# FTIR ML Solver v4.1 — Local Training

**Run order:**
- Cell 0 — environment check (always run first)
- Cell 1 — install package (editable local install)
- Cell 2 — build manifest
- Cell 3 — **Run A** control (8 epochs, no prior features)
- Cell 4 — **Run B** main (32 epochs, prior features) ← the important one
- Cell 5 — training status dashboard (run any time to see progress)
- Cell 6 — Run C optional comparison (32 epochs, trace_frac=0.10)
- Cell 7 — compare all completed runs
- Cell 8 — view MAE plots
- Cell 9 — show best checkpoint path

**Device:** auto-detected (CUDA > MPS > CPU). On a 96-core CPU machine, set `OMP_NUM_THREADS` and `MKL_NUM_THREADS` to your core count before launching Jupyter for best throughput.

In [ ]:
# Cell 0 — Environment check (run this first every session)
import sys, os, subprocess, platform
from pathlib import Path

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python:   {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")

try:
    import torch
    print(f"PyTorch:  {torch.__version__}")

    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"Device:   CUDA — {name}  ({vram:.1f} GB VRAM)")
    elif torch.backends.mps.is_available():
        print("Device:   MPS (Apple Silicon GPU)")
    else:
        cores = os.cpu_count()
        omp   = os.environ.get("OMP_NUM_THREADS", "not set")
        mkl   = os.environ.get("MKL_NUM_THREADS", "not set")
        print(f"Device:   CPU ({cores} logical cores)")
        print(f"  OMP_NUM_THREADS = {omp}  (set to {cores} for full throughput)")
        print(f"  MKL_NUM_THREADS = {mkl}")
        if omp == "not set":
            print(f"  → Run: export OMP_NUM_THREADS={cores} MKL_NUM_THREADS={cores}")
            print(f"    before launching Jupyter, or set them here:")
            print(f"    os.environ['OMP_NUM_THREADS'] = '{cores}'")
            print(f"    torch.set_num_threads({cores})")

except ImportError:
    print("PyTorch: not installed yet (run Cell 1 first)")

# RAM check (works on Linux; graceful on macOS)
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"RAM:      {vm.total / 1024**3:.0f} GB total, {vm.available / 1024**3:.0f} GB free")
except ImportError:
    r = subprocess.run(["free", "-h"], capture_output=True, text=True)
    if r.returncode == 0:
        lines = r.stdout.strip().splitlines()
        if len(lines) >= 2:
            print("RAM:     ", lines[1].split()[1], "total")

print()
print("Working directory:", Path.cwd())
print()
print("Existing checkpoints:")
for run in ("run_a", "run_b", "run_c"):
    ckpt = Path(f"outputs/checkpoints/{run}/best_model.pt")
    summ = Path(f"outputs/reports/{run}/training_summary.json")
    if ckpt.exists():
        import json
        if summ.exists():
            s = json.loads(summ.read_text())
            best = s.get("best_ref_epoch") or {}
            ep   = best.get("epoch", "?")
            ref  = best.get("ref_val_log_mae_mean", float("inf"))
            beat = best.get("species_beating_zero_baseline", "?")
            print(f"  {run}: epoch {ep}  ref_log_mae={ref:.4f}  beat_zero={beat}/11")
        else:
            print(f"  {run}: checkpoint found (no summary yet)")
    else:
        print(f"  {run}: not started")

In [ ]:
# Cell 1 — Install package (editable local install)
# Assumes the notebook is opened from the project root directory.
# If not, set PROJECT_ROOT below to the absolute path of the repo.
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd()  # change this if needed, e.g. Path("/path/to/FTIR_ML_fitting")

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(
        f"pyproject.toml not found in {PROJECT_ROOT}. "
        "Open the notebook from the project root, or set PROJECT_ROOT manually."
    )

os.chdir(PROJECT_ROOT)
print("Working directory:", PROJECT_ROOT)

!pip install -e . --no-deps -q

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    from ftir_analysis.constants import DEFAULT_TARGET_SPECIES, MODEL_VERSION
    print(f"✓ ftir_analysis {MODEL_VERSION} installed OK")
    print("  Target species:", DEFAULT_TARGET_SPECIES)
except ImportError as e:
    print(f"✗ Install failed: {e}")
    print("  Try: Kernel → Restart, then re-run this cell")

In [ ]:
# Cell 2 — Build manifest
from ftir_analysis.manifesting import build_manifest
m = build_manifest()
print(f"✓ Manifest: {len(m)} spectra, {m.species.nunique()} unique species")
print(m.groupby('species').size().to_string())

In [ ]:
# Cell 3 — Run A: control baseline (8 epochs, no prior features)
# Purpose: establish a baseline to compare Run B against.
# This run does NOT converge well — that is expected and is the point.
import torch, json, os
from pathlib import Path

# Auto-detect best available device
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
    # Use all available cores for CPU training
    cores = os.cpu_count()
    os.environ.setdefault("OMP_NUM_THREADS", str(cores))
    os.environ.setdefault("MKL_NUM_THREADS", str(cores))
    torch.set_num_threads(cores)

print("=" * 60)
print("RUN A  (control, no prior features, 8 epochs)")
print(f"Device: {DEVICE}")
print("=" * 60)

!python3 synthetic_generator.py \
    --n-samples 20000 \
    --seed 42 \
    --sampling-mode hybrid_v4 \
    --augmentation-profile mild \
    --hybrid-trace-fraction 0.15 \
    --min-active-species 1 \
    --max-active-species 4 \
    --out data/synthetic/spectra_v4_base.npz \
    --diagnostics-json outputs/reports/run_a/generation_diagnostics.json

!python3 train.py \
    --device {DEVICE} \
    --n-synthetic 20000 \
    --synthetic-npz data/synthetic/spectra_v4_base.npz \
    --epochs 8 \
    --batch-size 256 \
    --reference-weight 0.20 \
    --active-label-weight 4.0 \
    --inactive-label-weight 0.5 \
    --warmup-epochs 2.0 \
    --synthetic-sampling-mode hybrid_v4 \
    --synthetic-augmentation-profile mild \
    --hybrid-trace-fraction 0.15 \
    --min-active-species 1 \
    --max-active-species 4 \
    --checkpoint-dir outputs/checkpoints/run_a \
    --reports-dir outputs/reports/run_a \
    --log-every 2

# ---- Result summary ----
print()
print("-" * 40)
summary_path = Path("outputs/reports/run_a/training_summary.json")
if summary_path.exists():
    s = json.loads(summary_path.read_text())
    best = s.get("best_ref_epoch") or {}
    zero = best.get("zero_baseline_log_mae_mean", "?")
    ref  = best.get("ref_val_log_mae_mean", float("inf"))
    mix  = best.get("val_log_mae_mean", float("inf"))
    beat = best.get("species_beating_zero_baseline", "?")
    ep   = best.get("epoch", "?")
    grp  = best.get("group_log_mae_mean", {})
    maj  = (grp.get("all") or {}).get("major", float("inf"))
    trc  = (grp.get("all") or {}).get("trace", float("inf"))
    print(f"Run A result (best epoch {ep}):")
    print(f"  ref_log_mae  = {ref:.4f}   (zero baseline = {zero:.4f})")
    print(f"  mixed_log_mae= {mix:.4f}")
    print(f"  beat_zero    = {beat}/11")
    print(f"  major        = {maj:.4f}   trace = {trc:.4f}")
else:
    print("✗ No training_summary.json found — training may have failed.")
    ckpt = Path("outputs/checkpoints/run_a/best_model.pt")
    print(f"  Checkpoint exists: {ckpt.exists()}")

In [ ]:
# Cell 4 — Run B: main run (32 epochs, prior features)
# This is the important one.
# Prior features give the model pre-computed band integrals and
# template cosine similarity for each species — this is what makes training converge.
import torch, json, os
from pathlib import Path

# Auto-detect best available device
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
    cores = os.cpu_count()
    os.environ.setdefault("OMP_NUM_THREADS", str(cores))
    os.environ.setdefault("MKL_NUM_THREADS", str(cores))
    torch.set_num_threads(cores)

print("=" * 60)
print("RUN B  (prior features enabled, 32 epochs)")
print(f"Device: {DEVICE}")
print("=" * 60)

# Re-uses the spectra_v4_base.npz generated in Cell 3.
# If you skipped Cell 3, train.py will generate the synthetic data automatically.

!python3 train.py \
    --device {DEVICE} \
    --n-synthetic 20000 \
    --synthetic-npz data/synthetic/spectra_v4_base.npz \
    --epochs 32 \
    --batch-size 256 \
    --reference-weight 0.20 \
    --active-label-weight 4.0 \
    --inactive-label-weight 0.5 \
    --warmup-epochs 2.0 \
    --synthetic-sampling-mode hybrid_v4 \
    --synthetic-augmentation-profile mild \
    --hybrid-trace-fraction 0.15 \
    --min-active-species 1 \
    --max-active-species 4 \
    --use-prior-features \
    --checkpoint-dir outputs/checkpoints/run_b \
    --reports-dir outputs/reports/run_b \
    --log-every 1

# ---- Result summary ----
print()
print("-" * 40)
summary_path = Path("outputs/reports/run_b/training_summary.json")
if summary_path.exists():
    s = json.loads(summary_path.read_text())
    best = s.get("best_ref_epoch") or {}
    zero = best.get("zero_baseline_log_mae_mean", float("nan"))
    ref  = best.get("ref_val_log_mae_mean", float("inf"))
    mix  = best.get("val_log_mae_mean", float("inf"))
    beat = best.get("species_beating_zero_baseline", "?")
    ep   = best.get("epoch", "?")
    grp  = best.get("group_log_mae_mean", {})
    maj  = (grp.get("all") or {}).get("major", float("inf"))
    trc  = (grp.get("all") or {}).get("trace", float("inf"))
    zero_ref = best.get("zero_baseline_ref_log_mae_mean", float("nan"))
    print(f"Run B result (best epoch {ep} of 32):")
    print(f"  ref_log_mae   = {ref:.4f}   (ref zero baseline = {zero_ref:.4f})")
    print(f"  mixed_log_mae = {mix:.4f}   (all zero baseline = {zero:.4f})")
    print(f"  beat_zero     = {beat}/11")
    print(f"  major group   = {maj:.4f}   trace group = {trc:.4f}")
    if beat and isinstance(beat, int) and beat > 0:
        print(f"  ✓ Model beats zero baseline for {beat} species")
    else:
        print(f"  ✗ Model not yet beating zero baseline — more epochs may help")
else:
    print("✗ No training_summary.json found — training may have failed or was interrupted.")
    ckpt = Path("outputs/checkpoints/run_b/best_model.pt")
    print(f"  Checkpoint exists: {ckpt.exists()}")
    if ckpt.exists():
        print("  The checkpoint is there — check Cell 5 for partial results.")

In [ ]:
# Cell 5 — Training Status Dashboard
# Run this at ANY time to see what has completed without re-running training.
# Also shows per-species log-MAE from the best epoch of each run.
import json, torch
from pathlib import Path

TARGET_SPECIES = ['H2O','CO2','CO','NO','NO2','NH3','CH4','N2O','C2H4','HCN','HNCO']

def _load_summary(run: str) -> dict | None:
    p = Path.cwd() / f'outputs/reports/{run}/training_summary.json'
    if p.exists():
        return json.loads(p.read_text())
    return None

def _ckpt_exists(run: str) -> bool:
    return (Path.cwd() / f'outputs/checkpoints/{run}/best_model.pt').exists()

print("=" * 60)
print("TRAINING STATUS DASHBOARD")
print("=" * 60)

for run in ('run_a', 'run_b', 'run_c'):
    label = run.upper().replace('_', ' ')
    s = _load_summary(run)
    ckpt = _ckpt_exists(run)
    print(f"\n{label}")
    print("-" * 30)
    if s is None:
        print(f"  Status: {'checkpoint only (interrupted?)' if ckpt else 'not started'}")
        continue

    best_ref  = s.get('best_ref_epoch') or {}
    best_mix  = s.get('best_mixed_epoch') or {}

    for tag, data in [("Best ref epoch", best_ref), ("Best mixed epoch", best_mix)]:
        ep   = data.get('epoch', '?')
        ref  = data.get('ref_val_log_mae_mean')
        mix  = data.get('val_log_mae_mean', float('inf'))
        beat = data.get('species_beating_zero_baseline', '?')
        zero = data.get('zero_baseline_log_mae_mean', float('nan'))
        zero_ref = data.get('zero_baseline_ref_log_mae_mean', float('nan'))
        ref_str = f"{ref:.4f}" if ref is not None else "n/a"
        zero_ref_str = f"{zero_ref:.4f}" if zero_ref == zero_ref else "n/a"  # nan check
        print(f"  {tag}: epoch {ep}")
        print(f"    ref_log_mae   = {ref_str}  (zero baseline = {zero_ref_str})")
        print(f"    mixed_log_mae = {mix:.4f}   (zero baseline = {zero:.4f})")
        print(f"    beat_zero     = {beat}/11")
        grp = data.get('group_log_mae_mean', {})
        for split in ('all', 'ref'):
            sg = grp.get(split) or {}
            maj = sg.get('major', float('inf'))
            trc = sg.get('trace', float('inf'))
            if maj < float('inf'):
                print(f"    {split}: major={maj:.4f}  trace={trc:.4f}")

In [ ]:
# Cell 6 — Run C: comparison (32 epochs, prior features, trace_frac=0.10)
# Optional — run only after Run B completes.
# Tests whether reducing trace-species fraction improves major-species accuracy.
import torch, json, os
from pathlib import Path

# Auto-detect best available device
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
    cores = os.cpu_count()
    os.environ.setdefault("OMP_NUM_THREADS", str(cores))
    os.environ.setdefault("MKL_NUM_THREADS", str(cores))
    torch.set_num_threads(cores)

print("=" * 60)
print("RUN C  (prior features, trace_frac=0.10, 32 epochs)")
print(f"Device: {DEVICE}")
print("=" * 60)

!python3 synthetic_generator.py \
    --n-samples 20000 \
    --seed 42 \
    --sampling-mode hybrid_v4 \
    --augmentation-profile mild \
    --hybrid-trace-fraction 0.10 \
    --min-active-species 1 \
    --max-active-species 4 \
    --out data/synthetic/spectra_v4_trace10.npz \
    --diagnostics-json outputs/reports/run_c/generation_diagnostics.json

!python3 train.py \
    --device {DEVICE} \
    --n-synthetic 20000 \
    --synthetic-npz data/synthetic/spectra_v4_trace10.npz \
    --epochs 32 \
    --batch-size 256 \
    --reference-weight 0.20 \
    --active-label-weight 4.0 \
    --inactive-label-weight 0.5 \
    --warmup-epochs 2.0 \
    --synthetic-sampling-mode hybrid_v4 \
    --synthetic-augmentation-profile mild \
    --hybrid-trace-fraction 0.10 \
    --min-active-species 1 \
    --max-active-species 4 \
    --use-prior-features \
    --checkpoint-dir outputs/checkpoints/run_c \
    --reports-dir outputs/reports/run_c \
    --log-every 1

# ---- Result summary ----
print()
print("-" * 40)
summary_path = Path("outputs/reports/run_c/training_summary.json")
if summary_path.exists():
    s = json.loads(summary_path.read_text())
    best = s.get("best_ref_epoch") or {}
    ref  = best.get("ref_val_log_mae_mean", float("inf"))
    mix  = best.get("val_log_mae_mean", float("inf"))
    beat = best.get("species_beating_zero_baseline", "?")
    ep   = best.get("epoch", "?")
    grp  = best.get("group_log_mae_mean", {})
    maj  = (grp.get("all") or {}).get("major", float("inf"))
    trc  = (grp.get("all") or {}).get("trace", float("inf"))
    print(f"Run C result (best epoch {ep} of 32):")
    print(f"  ref_log_mae   = {ref:.4f}")
    print(f"  mixed_log_mae = {mix:.4f}")
    print(f"  beat_zero     = {beat}/11")
    print(f"  major group   = {maj:.4f}   trace group = {trc:.4f}")
else:
    print("✗ No summary found — training may have been interrupted.")

In [ ]:
# Cell 7 — Compare completed runs
import torch, json
from pathlib import Path

def _find(rel):
    p = Path.cwd() / rel
    return p if p.exists() else None

def _load_run(label, run_name):
    ckpt = _find(f'outputs/checkpoints/{run_name}/best_model.pt')
    sp   = _find(f'outputs/reports/{run_name}/training_summary.json')
    if ckpt is None:
        return None
    summary = json.loads(sp.read_text()) if sp else {}
    return {
        'label': label, 'run_name': run_name, 'ckpt': ckpt,
        'selected': summary.get('best_ref_epoch') or {},
        'best_mixed': summary.get('best_mixed_epoch'),
    }

def _score(d):
    ref  = float(d.get('ref_val_log_mae_mean') or float('inf'))
    val  = float(d.get('val_log_mae_mean') or float('inf'))
    beat = int(d.get('species_beating_zero_baseline') or 0)
    return ref, val, beat

def _group(d, split, group):
    return float((d.get('group_log_mae_mean') or {}).get(split, {}).get(group, float('inf')) or float('inf'))

runs = [r for r in [_load_run('A','run_a'), _load_run('B','run_b'), _load_run('C','run_c')] if r]
if not runs:
    print("No completed runs found. Run at least Cell 3 or Cell 4 first.")
else:
    ctrl = next((r for r in runs if r['label'] == 'A'), None)
    ctrl_ref = float(ctrl['selected'].get('ref_val_log_mae_mean') or float('inf')) if ctrl else float('inf')
    ctrl_maj = _group(ctrl['selected'], 'all', 'major') if ctrl else float('inf')

    print("=" * 60)
    print("RUN COMPARISON")
    print("=" * 60)
    scored = []
    for r in runs:
        sel = r['selected']
        ref, val, beat = _score(sel)
        maj_all  = _group(sel, 'all', 'major')
        trc_all  = _group(sel, 'all', 'trace')
        maj_ref  = _group(sel, 'ref', 'major')
        trc_ref  = _group(sel, 'ref', 'trace')
        zero     = float(sel.get('zero_baseline_log_mae_mean') or float('nan'))
        zero_ref = float(sel.get('zero_baseline_ref_log_mae_mean') or float('nan'))
        ep       = sel.get('epoch', '?')

        guard = True
        vs_ctrl = ""
        if ctrl and r['label'] != 'A':
            gain = ctrl_ref - ref
            regress = maj_all - ctrl_maj
            guard = not (gain < 0.02 and regress > 0.05)
            vs_ctrl = f"  vs A: ref_gain={gain:+.4f}  major_regression={regress:+.4f}  guardrail={'OK' if guard else 'FAIL'}"

        ref_str  = f"{ref:.4f}" if ref < float('inf') else "n/a"
        zero_ref_str = f"{zero_ref:.4f}" if zero_ref == zero_ref else "n/a"

        print(f"\nRun {r['label']}  (best epoch {ep})")
        print(f"  ref_log_mae   = {ref_str}  (zero = {zero_ref_str})")
        print(f"  mixed_log_mae = {val:.4f}   (zero = {zero:.4f})")
        print(f"  beat_zero     = {beat}/11")
        print(f"  major={maj_all:.4f}  trace={trc_all:.4f}  (ref: major={maj_ref:.4f}  trace={trc_ref:.4f})")
        if vs_ctrl:
            print(vs_ctrl)
        scored.append((guard, ref, val, -beat, r))

    viable = [x for x in scored if x[0]] or scored
    _, br, bv, _, winner = min(viable, key=lambda x: (x[1], x[2], x[3]))
    print()
    print("=" * 60)
    print(f"Winner: Run {winner['label']}  ref={br:.4f}  mixed={bv:.4f}")
    print(f"Checkpoint: {winner['ckpt']}")

In [ ]:
# Cell 8 — View per-species MAE plots for all runs
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

roots = [Path.cwd(), Path('/content/ftir')]
runs  = ['run_a', 'run_b', 'run_c']

shown = 0
for run in runs:
    plots = []
    for root in roots:
        plots += list(root.glob(f'outputs/reports/{run}/mae_per_species_*.png'))
    if not plots:
        continue
    latest = sorted(plots, key=lambda p: p.stat().st_mtime)[-1]
    img = mpimg.imread(str(latest))
    plt.figure(figsize=(12, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'{run.upper()} — {latest.name}', fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f"Loaded: {latest}")
    shown += 1

if shown == 0:
    print("No MAE plots found yet. Run at least Cell 3 or Cell 4 first.")

In [ ]:
# Cell 9 — Show best checkpoint path
# Prints the path to the best checkpoint so you can copy it for inference.
import json
from pathlib import Path

def _find(rel):
    p = Path.cwd() / rel
    return p if p.exists() else None

def _load_run(label, run_name):
    ckpt = _find(f'outputs/checkpoints/{run_name}/best_model.pt')
    sp   = _find(f'outputs/reports/{run_name}/training_summary.json')
    if ckpt is None:
        return None
    summary = json.loads(sp.read_text()) if sp else {}
    return {'label': label, 'ckpt': ckpt,
            'selected': summary.get('best_ref_epoch') or {}}

runs = [r for r in [_load_run('A','run_a'), _load_run('B','run_b'), _load_run('C','run_c')] if r]
if not runs:
    raise FileNotFoundError("No checkpoints found. Run Cell 3 or Cell 4 first.")

def _score(r):
    ref  = float(r['selected'].get('ref_val_log_mae_mean') or float('inf'))
    val  = float(r['selected'].get('val_log_mae_mean') or float('inf'))
    beat = int(r['selected'].get('species_beating_zero_baseline') or 0)
    return ref, val, -beat

winner = min(runs, key=_score)
ref, val, neg_beat = _score(winner)
ckpt_path = winner['ckpt'].resolve()
meta_path = ckpt_path.with_suffix('.meta.json')

print(f"Best run:   Run {winner['label']}")
print(f"  ref_log_mae = {ref:.4f}   mixed = {val:.4f}   beat_zero = {-neg_beat}/11")
print()
print(f"Checkpoint: {ckpt_path}")
if meta_path.exists():
    print(f"Metadata:   {meta_path}")
print()
print("Use for inference:")
print(f"  python inference.py --data-dir path/to/spc_files/ --checkpoint {ckpt_path}")